# Complete final-report computation

Runs the complete analysis from this repository, including 999 state-bootstrap resamples, and verifies the final report results.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPOSITORY = 'https://github.com/ronyspada2025/walsh-msc-capstone.git'
# The update script adjusts this default when publishing an alternate branch.
# CAPSTONE_REF selects the Git branch or tag to run.
PROJECT_REF = os.environ.get('CAPSTONE_REF', 'main')
ROOT = Path.cwd()
if not (ROOT/'final_pipeline.py').exists() and (ROOT.parent/'final_pipeline.py').exists():
    ROOT=ROOT.parent
if not (ROOT/'final_pipeline.py').exists():
    ROOT=Path.cwd()/'walsh-msc-capstone'
    if ROOT.exists():
        raise RuntimeError('Destination already exists. Restart in a clean runtime or open its repository notebook.')
    subprocess.run(['git','clone','--branch',PROJECT_REF,'--single-branch',REPOSITORY,str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
sys.path.insert(0,str(ROOT))
print('Repository:',ROOT)
print(subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip())


In [ ]:
env=os.environ.copy()
env.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',LOKY_MAX_CPU_COUNT='4')
subprocess.run([sys.executable,'final_pipeline.py','--jobs','4'],cwd=ROOT,env=env,check=True)
subprocess.run([sys.executable,'scripts/verify_project.py'],cwd=ROOT,check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
r=json.loads((ROOT/'reports/tables/headline_results.json').read_text())
display(pd.read_csv(ROOT/'reports/tables/table14_predictive_performance.csv'))
display(Image(filename=str(ROOT/'reports/figures/figure11_permutation_importance.png')))
print((ROOT/'reports/report_comparison.md').read_text())
